In [5]:
import torch
import os

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("GPU Count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected")

PyTorch Version: 2.14.1+cpu
CUDA Available: False
CUDA Version: None
GPU Count: 0
GPU not detected


Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime


In [1]:
!nvidia-smi

import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not detected")

Fri Nov  6 15:38:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.


In [3]:
!pip install -q "huggingface-hub>=0.25.0,<2.0"

In [4]:
import huggingface_hub
import datasets
import torch

print("Hugging Face Hub:", huggingface_hub.__version__)
print("Datasets:", datasets.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Hugging Face Hub: 1.33.0
Datasets: 4.8.5
CUDA Available: True
GPU: Tesla T4


In [5]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("FP16 supported:", torch.cuda.is_available())
print("BF16 supported:", torch.cuda.is_bf16_supported())

Transformers: 5.22.1
FP16 supported: True
BF16 supported: True


In [6]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("FP16 supported:", torch.cuda.is_available())
print("BF16 supported:", torch.cuda.is_bf16_supported())

Transformers: 5.22.1
FP16 supported: True
BF16 supported: True


In [7]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Gemma 4 loaded successfully!")

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.6k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.95k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Gemma 4 loaded successfully!


In [8]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": """
                Write a Python function to find the largest
                number in a list.

                Explain the code step by step.
                Also provide one example.
                """
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    do_sample=False
)

response = processor.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

user
Write a Python function to find the largest
                number in a list.

                Explain the code step by step.
                Also provide one example.
model
Here is the Python function to find the largest number in a list, along with a step-by-step explanation and an example.

## Python Function

```python
def find_largest_number(numbers):
  """
  Finds and returns the largest number in a given list of numbers.

  Args:
    numbers: A list of integers or floats.

  Returns:
    The largest number in the list, or None if the list is empty.
  """
  # 1. Handle the edge case of an empty list
  if not numbers:
    return None

  # 2. Initialize the 'largest' variable
  # Assume the first element is the largest initially
  largest = numbers[0]

  # 3. Iterate through the rest of the list
  # Start from the second element (index 1) since we already checked index 0
  for number in numbers[1:]:



In [9]:
bug_prompt = """
Find the bug in this Python function.

def find_largest_number(numbers):
    largest = 0

    for number in numbers:
        if number > largest:
            largest = number

    return largest

The function should work with both positive and negative numbers.

Explain:
1. What is the bug?
2. Why does it happen?
3. How can you fix it?
4. Provide the corrected code.
"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": bug_prompt
            }
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False
)

response = processor.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

user
Find the bug in this Python function.

def find_largest_number(numbers):
    largest = 0

    for number in numbers:
        if number > largest:
            largest = number

    return largest

The function should work with both positive and negative numbers.

Explain:
1. What is the bug?
2. Why does it happen?
3. How can you fix it?
4. Provide the corrected code.
model
Here is the analysis and fix for the bug in the provided Python function.

## Bug Analysis

### 1. What is the bug?

The bug is that the function incorrectly initializes the `largest` variable to `0`. This causes the function to fail when the input list `numbers` contains only negative numbers, or when the largest number in the list is negative (e.g., in `[-10, -5, -20]`, the function will incorrectly return `0` instead of `-5`).

### 2. Why does it happen?

The initialization `largest = 0` sets a baseline assumption that the largest number found will be greater than or equal to zero.

*   **Scenario 1 (Positive 

In [11]:
def find_largest_number(numbers):
    if not numbers:
        return None

    largest = numbers[0]

    for number in numbers[1:]:
        if number > largest:
            largest = number

    return largest

In [12]:
def find_largest_number(numbers):
    if not numbers:
        return None

    largest = numbers[0]

    for number in numbers[1:]:
        if number > largest:
            largest = number

    return largest


test_cases = [
    [10, 25, 5, 40, 15],
    [-10, -5, -20],
    [-100, -200, -50],
    [0, -5, -10],
    []
]

for test in test_cases:
    print("Input:", test)
    print("Output:", find_largest_number(test))
    print("---")

Input: [10, 25, 5, 40, 15]
Output: 40
---
Input: [-10, -5, -20]
Output: -5
---
Input: [-100, -200, -50]
Output: -50
---
Input: [0, -5, -10]
Output: 0
---
Input: []
Output: None
---


In [13]:
from pathlib import Path

project = Path("bookstore")
project.mkdir(exist_ok=True)

files = {
    "main.py": '''
from orders import place_order

def run():
    return place_order("BOOK", 2, "SAVE10")
''',

    "orders.py": '''
from pricing import calculate_total
from discounts import get_discount

def place_order(product_id, quantity, coupon):
    unit_price = {"BOOK": 100}.get(product_id, 0)
    discount_rate = get_discount(coupon)

    return calculate_total(
        unit_price, quantity, discount_rate
    )
''',

    "discounts.py": '''
def get_discount(coupon):
    if coupon == "SAVE10":
        return 10
    return 0
''',

    "pricing.py": '''
def calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    discount = subtotal * discount_rate
    return subtotal - discount
'''
}

for filename, content in files.items():
    (project / filename).write_text(content)

print("Repository created successfully!")

for file in project.iterdir():
    print(file.name)

Repository created successfully!
main.py
pricing.py
orders.py
discounts.py


In [14]:
repository_context = ""

for filename in ["main.py", "orders.py", "discounts.py", "pricing.py"]:
    content = (project / filename).read_text()
    repository_context += f"\nFILE: {filename}\n{content}\n"

bug_prompt = f"""
You are a Python repository debugging agent.

Here is the repository:

{repository_context}

BUG REPORT:
A customer buys 2 books at 100 each and applies
the SAVE10 coupon.

Expected total: 180
Actual total: 0

TASK:
1. Identify the most likely buggy file.
2. Identify the exact function and line.
3. Explain the root cause.
4. Suggest a minimal fix.
5. Explain how the files depend on each other.

Do not modify the code.
"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": bug_prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False
)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)


## Debugging Analysis

**1. Identify the most likely buggy file.**
`pricing.py` or `orders.py` (due to how the discount rate is used).

**2. Identify the exact function and line.**
The issue lies in the calculation within `pricing.py`:
```python
# File: pricing.py
def calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    discount = subtotal * discount_rate  # <-- Potential issue here
    return subtotal - discount
```

**3. Explain the root cause.**
The `get_discount` function in `discounts.py` returns `10` when the coupon is "SAVE10". This value represents a percentage (10%).

In `pricing.py`, the calculation for the discount is:
`discount = subtotal * discount_rate`

If `unit_price` is 100 and `quantity` is 2, `subtotal` is 200.
If `discount_rate` is 10 (from `get_discount`), the calculated `discount` becomes $200 \times 10 = 2000$.
The final result is $200 - 2000 = -1800$.

However, the expected total is 180. This implies


In [15]:
def get_discount(coupon):
    if coupon == "SAVE10":
        return 10
    return 0

In [16]:
def calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    discount = subtotal * discount_rate
    return subtotal - discount

In [17]:
def calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity

    discount = subtotal * (discount_rate / 100)

    return subtotal - discount

In [18]:
import sys

sys.path.insert(0, str(project.resolve()))

from main import run

result = run()

print("Actual result:", result)
print("Expected result:", 180)
print("Test passed:", result == 180)

Actual result: -1800
Expected result: 180
Test passed: False


In [19]:
dependency_graph = """
Repository Dependency Graph:

main.py
  -> orders.py

orders.py
  -> pricing.py
  -> discounts.py

pricing.py
  -> No imports

discounts.py
  -> No imports
"""

print(dependency_graph)


Repository Dependency Graph:

main.py
  -> orders.py

orders.py
  -> pricing.py
  -> discounts.py

pricing.py
  -> No imports

discounts.py
  -> No imports



In [20]:
graph_prompt = f"""
You are a Python repository debugging agent.

SOURCE CODE:

{repository_context}

DEPENDENCY GRAPH:

{dependency_graph}

BUG REPORT:
A customer buys 2 books at 100 each and applies
the SAVE10 coupon.

Expected total: 180
Actual total: -1800

TASK:
1. Identify the single most likely buggy file.
2. Identify the exact function and problematic line.
3. Explain how the dependency graph helps locate the bug.
4. Explain the root cause.
5. Suggest a minimal fix.

Return your reasoning clearly.
"""

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": graph_prompt}
        ]
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=350,
    do_sample=False
)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

## Debugging Analysis

Here is the analysis of the provided code and bug report.

### 1. Identify the single most likely buggy file.

**`pricing.py`**

### 2. Identify the exact function and problematic line.

**Function:** `calculate_total`
**Problematic Line:** `return subtotal - discount`

### 3. Explain how the dependency graph helps locate the bug.

The dependency graph shows the flow of execution:
`main.py` $\rightarrow$ `orders.py` $\rightarrow$ (`pricing.py` and `discounts.py`).

The bug report indicates that the final calculation results in a large negative number (Actual total: -1800), suggesting an incorrect operation within the pricing logic. Since the inputs (`unit_price`, `quantity`, `discount_rate`) are derived from the preceding steps (`orders.py` and `discounts.py`), the error is most likely in the function responsible for the final arithmetic calculation, which is `calculate_total` in `pricing.py`.

### 4. Explain the root cause.

The calculation in `pricing.py` is:
1

In [21]:
distractors = {
    "inventory.py": '''
def check_stock(product_id):
    stock = {"BOOK": 50}
    return stock.get(product_id, 0)
''',

    "shipping.py": '''
def calculate_shipping(quantity):
    if quantity >= 5:
        return 0
    return 50
''',

    "tax.py": '''
def calculate_tax(subtotal, tax_rate):
    return subtotal * (tax_rate / 100)
''',

    "catalog.py": '''
def get_product_price(product_id):
    products = {"BOOK": 100}
    return products.get(product_id, 0)
''',

    "validation.py": '''
def validate_quantity(quantity):
    return quantity > 0
''',

    "notifications.py": '''
def send_order_notification(order_id):
    return f"Order {order_id} received"
'''
}

for filename, content in distractors.items():
    (project / filename).write_text(content)

print("Additional files created:")
for filename in distractors:
    print("-", filename)

Additional files created:
- inventory.py
- shipping.py
- tax.py
- catalog.py
- validation.py
- notifications.py


In [22]:
repository_context_v2 = ""

for file_path in sorted(project.glob("*.py")):
    content = file_path.read_text()

    repository_context_v2 += (
        f"\nFILE: {file_path.name}\n"
        f"{content}\n"
    )

print("Total Python files:", len(list(project.glob("*.py"))))
print(repository_context_v2)

Total Python files: 10

FILE: catalog.py

def get_product_price(product_id):
    products = {"BOOK": 100}
    return products.get(product_id, 0)


FILE: discounts.py

def get_discount(coupon):
    if coupon == "SAVE10":
        return 10
    return 0


FILE: inventory.py

def check_stock(product_id):
    stock = {"BOOK": 50}
    return stock.get(product_id, 0)


FILE: main.py

from orders import place_order

def run():
    return place_order("BOOK", 2, "SAVE10")


FILE: notifications.py

def send_order_notification(order_id):
    return f"Order {order_id} received"


FILE: orders.py

from pricing import calculate_total
from discounts import get_discount

def place_order(product_id, quantity, coupon):
    unit_price = {"BOOK": 100}.get(product_id, 0)
    discount_rate = get_discount(coupon)

    return calculate_total(
        unit_price, quantity, discount_rate
    )


FILE: pricing.py

def calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    d

In [23]:
print("Total Python files:", len(list(project.glob("*.py"))))

print("\nRepository files:")
for file_path in sorted(project.glob("*.py")):
    print("-", file_path.name)

print("\nContext size:", len(repository_context_v2), "characters")

Total Python files: 10

Repository files:
- catalog.py
- discounts.py
- inventory.py
- main.py
- notifications.py
- orders.py
- pricing.py
- shipping.py
- tax.py
- validation.py

Context size: 1326 characters


In [25]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print("Input tokens:", inputs["input_ids"].shape[-1])

CUDA available: True
GPU: Tesla T4
GPU memory allocated: 9.53 GB
Input tokens: 540


In [26]:
outputs = model.generate(
    **inputs,
    max_new_tokens=120,
    do_sample=False,
    use_cache=True
)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

## Debugging Analysis

**1. Identify the single most likely buggy file.**
`pricing.py`

**2. Identify the exact function and faulty line.**
Function: `calculate_total`
Faulty Line: `return subtotal - discount` (or the calculation leading to it)

**3. Explain the root cause.**
The bug occurs because the `discount_rate` returned by `get_discount("SAVE10")` is `10` (representing 10%), but the calculation treats this value as a multiplier (100% discount


In [28]:
dependency_graph = {
    "main.py": ["orders.py"],
    "orders.py": ["pricing.py", "discounts.py"],
    "pricing.py": [],
    "discounts.py": [],
    "catalog.py": [],
    "inventory.py": [],
    "shipping.py": [],
    "tax.py": [],
    "validation.py": [],
    "notifications.py": []
}

for file, dependencies in dependency_graph.items():
    print(f"{file} --> {dependencies}")

main.py --> ['orders.py']
orders.py --> ['pricing.py', 'discounts.py']
pricing.py --> []
discounts.py --> []
catalog.py --> []
inventory.py --> []
shipping.py --> []
tax.py --> []
validation.py --> []
notifications.py --> []


In [29]:
print("Total files:", len(dependency_graph))

print(
    "Files connected to orders.py:",
    dependency_graph["orders.py"]
)

print(
    "Files depending on pricing.py:",
    [
        file for file, dependencies in dependency_graph.items()
        if "pricing.py" in dependencies
    ]
)

Total files: 10
Files connected to orders.py: ['pricing.py', 'discounts.py']
Files depending on pricing.py: ['orders.py']


In [30]:
graph_text = "\nDEPENDENCY GRAPH:\n"

for file, dependencies in dependency_graph.items():
    graph_text += f"{file} -> {dependencies}\n"

graph_augmented_context = (
    repository_context_v2 + graph_text
)

print("Repository context:", len(repository_context_v2), "characters")
print("Graph context:", len(graph_text), "characters")
print("Combined context:", len(graph_augmented_context), "characters")

print("\nGraph preview:")
print(graph_text)

Repository context: 1326 characters
Graph context: 234 characters
Combined context: 1560 characters

Graph preview:

DEPENDENCY GRAPH:
main.py -> ['orders.py']
orders.py -> ['pricing.py', 'discounts.py']
pricing.py -> []
discounts.py -> []
catalog.py -> []
inventory.py -> []
shipping.py -> []
tax.py -> []
validation.py -> []
notifications.py -> []



In [31]:
graph_prompt = f"""
You are an expert Python debugging assistant.

Analyze the following Python repository.

REPOSITORY:
{graph_augmented_context}

TASK:
1. Identify the single most likely buggy file.
2. Identify the exact function and faulty line.
3. Explain the root cause.
4. Explain how the dependency graph helps locate the bug.
5. Explain why other files are less likely.
6. Suggest the minimal code fix.

Important:
- Do not guess.
- Use evidence from the provided source code.
- Identify the exact faulty statement, not just the function.
- The dependency graph is additional context, not proof of a bug.
"""

inputs = processor(
    text=graph_prompt,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", inputs["input_ids"].shape[-1])

Input tokens: 626


In [32]:
outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    do_sample=False,
    use_cache=True
)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- Do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.

- do not guess.




In [33]:
graph_prompt_clean = f"""
Analyze this Python repository and find the bug.

Repository source code and dependency graph:
{graph_augmented_context}

Answer using exactly these six sections:

1. Buggy file:
2. Function:
3. Exact faulty statement:
4. Root cause:
5. Dependency graph evidence:
6. Minimal fix:

Use specific code evidence.
If uncertain, explain what cannot be determined.
"""

messages = [
    {
        "role": "user",
        "content": graph_prompt_clean
    }
]

formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=formatted_prompt,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", inputs["input_ids"].shape[-1])

Input tokens: 581


In [34]:
outputs = model.generate(
    **inputs,
    max_new_tokens=220,
    do_sample=False,
    repetition_penalty=1.1,
    use_cache=True
)

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

1. Buggy file: `orders.py`
2. Function: `place_order`
3. Exact faulty statement: `unit_price = {"BOOK": 100}.get(product_id, 0)`
4. Root cause: The logic for determining the unit price is hardcoded within `orders.py` instead of utilizing the dedicated function in `catalog.py` (`get_product_price`), leading to potential inconsistency or maintenance issues if product prices change. Furthermore, the implementation in `orders.py` duplicates the logic found in `catalog.py`.
5. Dependency graph evidence:
    * `orders.py` imports `calculate_total` from `pricing.py` and `get_discount` from `discounts.py`.
    * Although `catalog.py` exists with `get_product_price(product_id)`, `orders.py` does not import or use it. Instead, it uses a hardcoded dictionary: `unit_price = {"BOOK": 100}.get


In [35]:
baseline_prompt_fair = f"""
Analyze this Python repository and find the bug.

Repository source code:
{repository_context_v2}

Answer using exactly these six sections:

1. Buggy file:
2. Function:
3. Exact faulty statement:
4. Root cause:
5. Repository evidence:
6. Minimal fix:

Use specific code evidence.
If uncertain, explain what cannot be determined.
"""

messages = [
    {
        "role": "user",
        "content": baseline_prompt_fair
    }
]

formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

baseline_inputs = processor(
    text=formatted_prompt,
    return_tensors="pt"
).to(model.device)

print("Baseline input tokens:",
      baseline_inputs["input_ids"].shape[-1])

Baseline input tokens: 497


In [36]:
baseline_outputs = model.generate(
    **baseline_inputs,
    max_new_tokens=220,
    do_sample=False,
    repetition_penalty=1.1,
    use_cache=True
)

baseline_response = processor.decode(
    baseline_outputs[0][baseline_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(baseline_response)

Here is the analysis of the provided Python repository to find a potential bug.

1. Buggy file: `orders.py`
2. Function: `place_order`
3. Exact faulty statement: `unit_price = {"BOOK": 100}.get(product_id, 0)`
4. Root cause: The logic for determining the base unit price is hardcoded within `place_order` instead of relying on a centralized or consistent pricing mechanism defined elsewhere (like in `catalog.py`). This leads to inconsistency if other parts of the system rely on `catalog.py`. Specifically, `catalog.py` defines `get_product_price`, which should ideally be used here.
5. Repository evidence:
   - In `orders.py`:
     ```python
     def place_order(product_id, quantity, coupon):
         unit_price = {"BOOK": 100}.get(product_id, 0) # Hardcoded price lookup
         discount_rate = get_discount(


In [37]:
from collections import deque

def retrieve_related_files(start_file, graph):
    visited = set()
    queue = deque([start_file])

    while queue:
        current = queue.popleft()

        if current in visited:
            continue

        visited.add(current)

        for dependency in graph.get(current, []):
            if dependency not in visited:
                queue.append(dependency)

    return sorted(visited)


relevant_files = retrieve_related_files(
    "main.py",
    dependency_graph
)

print("Retrieved files:", len(relevant_files))

for file in relevant_files:
    print(file)

Retrieved files: 4
discounts.py
main.py
orders.py
pricing.py


In [38]:
retrieved_context = ""

for filename in relevant_files:
    file_path = project / filename

    retrieved_context += (
        f"\nFILE: {filename}\n"
        f"{file_path.read_text()}\n"
    )

print("Full repository files:", 10)
print("Retrieved files:", len(relevant_files))
print("Retrieved context characters:", len(retrieved_context))

Full repository files: 10
Retrieved files: 4
Retrieved context characters: 704


In [39]:
retrieval_prompt = f"""
Analyze the following retrieved Python files.

REPOSITORY CODE:
{retrieved_context}

TASK:
1. Identify the actual buggy file.
2. Identify the function containing the bug.
3. Identify the exact faulty statement.
4. Explain the root cause using code evidence.
5. Provide the minimal correction.

Important:
A design improvement is not necessarily a bug.
Find an actual incorrect calculation or behavior.
Do not assume that hardcoded values are bugs.
"""

messages = [
    {
        "role": "user",
        "content": retrieval_prompt
    }
]

formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

retrieval_inputs = processor(
    text=formatted_prompt,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", retrieval_inputs["input_ids"].shape[-1])

Input tokens: 311


In [40]:
retrieval_prompt = f"""
Analyze the following retrieved Python files.

REPOSITORY CODE:
{retrieved_context}

TASK:
1. Identify the actual buggy file.
2. Identify the function containing the bug.
3. Identify the exact faulty statement.
4. Explain the root cause using code evidence.
5. Provide the minimal correction.

Important:
A design improvement is not necessarily a bug.
Find an actual incorrect calculation or behavior.
Do not assume that hardcoded values are bugs.
"""

messages = [
    {
        "role": "user",
        "content": retrieval_prompt
    }
]

formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

retrieval_inputs = processor(
    text=formatted_prompt,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", retrieval_inputs["input_ids"].shape[-1])

Input tokens: 311


In [41]:
retrieval_outputs = model.generate(
    **retrieval_inputs,
    max_new_tokens=220,
    do_sample=False,
    repetition_penalty=1.1,
    use_cache=True
)

retrieval_response = processor.decode(
    retrieval_outputs[0][retrieval_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(retrieval_response)

Here is the analysis of the provided Python files:

## Analysis

### 1. Identify the actual buggy file.
The buggy file is **`pricing.py`**.

### 2. Identify the function containing the bug.
The function containing the bug is `calculate_total` in `pricing.py`.

### 3. Identify the exact faulty statement.
The faulty statement is:
```python
    discount = subtotal * discount_rate
```

### 4. Explain the root cause using code evidence.
The logic for calculating the final total is flawed because it calculates the discount based on the *entire subtotal* multiplied by the `discount_rate`, rather than applying the discount rate as a percentage reduction to the subtotal.

In this specific scenario, the `get_discount` function returns an integer (e.g., 10 for "SAVE10"). When this integer is passed to `calculate_total`, it is treated as a multiplier instead of a percentage factor (which should be divided by 100).


In [42]:
def original_calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    discount = subtotal * discount_rate
    return subtotal - discount


def corrected_calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    discount = subtotal * (discount_rate / 100)
    return subtotal - discount


print("Original result:",
      original_calculate_total(100, 2, 10))

print("Corrected result:",
      corrected_calculate_total(100, 2, 10))

print("Expected result: 180")

Original result: -1800
Corrected result: 180.0
Expected result: 180


In [43]:
test_cases = [
    (100, 2, 10, 180),
    (200, 3, 20, 480),
    (50, 4, 0, 200),
    (100, 1, 100, 0),
]

for price, quantity, rate, expected in test_cases:
    actual = corrected_calculate_total(price, quantity, rate)

    assert actual == expected, (
        f"Failed: expected {expected}, got {actual}"
    )

print("All tests passed!")

All tests passed!


In [44]:
import torch

seeds = [11, 22, 33, 44, 55]
repeated_results = []

for run_number, seed in enumerate(seeds, start=1):

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    outputs = model.generate(
        **retrieval_inputs,
        max_new_tokens=220,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        use_cache=True
    )

    response = processor.decode(
        outputs[0][retrieval_inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    repeated_results.append({
        "run": run_number,
        "seed": seed,
        "response": response
    })

    print(f"\n{'=' * 20} RUN {run_number} {'=' * 20}")
    print(response)

print("\nTotal completed runs:", len(repeated_results))


==================== RUN 1 ====================
Here is the analysis of the provided Python files:

## Analysis

**1. Identify the actual buggy file.**
The buggy file is **`pricing.py`**.

**2. Identify the function containing the bug.**
The function containing the bug is `calculate_total` in `pricing.py`.

**3. Identify the exact faulty statement.**
```python
    discount = subtotal * discount_rate
```

**4. Explain the root cause using code evidence.**
The `get_discount` function in `discounts.py` returns an integer (e.g., `10` for "SAVE10"). This value is passed as `discount_rate` to `calculate_total`.

In `pricing.py`:
*   If `unit_price` is 100 and `quantity` is 2, `subtotal` is $200$.
*   If `coupon` is "SAVE10", `discount_rate` is $10$.
*   The calculated `discount`

==================== RUN 2 ====================
Here is the analysis of the provided Python files to identify the bug.

## Analysis

1.  **`discounts.py`**: Defines a simple discount lookup.
    ```python
    def g

In [45]:
import random

all_files = sorted([
    file.name for file in project.glob("*.py")
])

random_files = random.Random(42).sample(all_files, 4)

print("Randomly selected files:")
for filename in random_files:
    print("-", filename)

print("\nTotal selected:", len(random_files))

Randomly selected files:
- discounts.py
- catalog.py
- notifications.py
- validation.py

Total selected: 4


In [46]:
random_context = ""

for filename in random_files:
    file_path = project / filename
    random_context += f"\nFILE: {filename}\n"
    random_context += file_path.read_text() + "\n"

print(random_context)
print("\nContext characters:", len(random_context))


FILE: discounts.py

def get_discount(coupon):
    if coupon == "SAVE10":
        return 10
    return 0


FILE: catalog.py

def get_product_price(product_id):
    products = {"BOOK": 100}
    return products.get(product_id, 0)


FILE: notifications.py

def send_order_notification(order_id):
    return f"Order {order_id} received"


FILE: validation.py

def validate_quantity(quantity):
    return quantity > 0



Context characters: 414


In [47]:
random_prompt = f"""
Analyze the following retrieved Python files.

REPOSITORY CODE:
{random_context}

TASK:
1. Identify the actual buggy file.
2. Identify the function containing the bug.
3. Identify the exact faulty statement.
4. Explain the root cause using code evidence.
5. Provide the minimal correction.

Important:
A design improvement is not necessarily a bug.
Do not invent missing code.
If the actual bug cannot be identified from the retrieved files,
clearly say so.
"""

messages = [
    {"role": "user", "content": random_prompt}
]

formatted_prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

random_inputs = processor(
    text=formatted_prompt,
    return_tensors="pt"
).to(model.device)

print("Input tokens:", random_inputs["input_ids"].shape[-1])

Input tokens: 239


In [48]:
random_outputs = model.generate(
    **random_inputs,
    max_new_tokens=220,
    do_sample=False,
    repetition_penalty=1.1,
    use_cache=True
)

random_response = processor.decode(
    random_outputs[0][random_inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(random_response)

Based on the provided Python files, I have analyzed them against the typical expectations of such modules.

**Analysis:**

*   **`discounts.py`**: `get_discount(coupon)` correctly returns 10 for "SAVE10" and 0 otherwise. This appears logically sound based on its implementation.
*   **`catalog.py`**: `get_product_price(product_id)` returns 100 for "BOOK" and 0 for any other ID. This is a valid dictionary lookup pattern.
*   **`notifications.py`**: `send_order_notification(order_id)` returns a formatted string. This is functional.
*   **`validation.py`**: `validate_quantity(quantity)` checks if the quantity is greater than zero (`quantity > 0`). This is a standard validation check.

**Conclusion:**

None of the provided files contain an obvious syntax error or a logical flaw that would cause a runtime crash or incorrect behavior *based solely on the code provided*. All functions perform the


In [49]:
import random
import torch

seeds = [7, 21, 42, 77, 99]

random_experiment_results = []

for run_no, seed in enumerate(seeds, start=1):

    # 1. Select 4 random files
    selected_files = random.Random(seed).sample(all_files, 4)

    # 2. Check whether actual buggy file is included
    bug_file_found = "pricing.py" in selected_files

    # 3. Build repository context
    context = ""

    for filename in selected_files:
        file_path = project / filename
        context += f"\nFILE: {filename}\n"
        context += file_path.read_text() + "\n"

    # 4. Create the same prompt
    prompt = f"""
Analyze the following retrieved Python files.

REPOSITORY CODE:
{context}

TASK:
1. Identify the actual buggy file.
2. Identify the function containing the bug.
3. Identify the exact faulty statement.
4. Explain the root cause using code evidence.
5. Provide the minimal correction.

Important:
A design improvement is not necessarily a bug.
Do not invent missing code.
If the actual bug cannot be identified from the retrieved files,
clearly say so.
"""

    messages = [{"role": "user", "content": prompt}]

    formatted_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    # 5. Generate Gemma response
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.1,
            use_cache=True
        )

    response = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    # 6. Save experiment results
    result = {
        "run": run_no,
        "seed": seed,
        "selected_files": selected_files,
        "bug_file_found": bug_file_found,
        "context_chars": len(context),
        "response": response
    }

    random_experiment_results.append(result)

    print(f"\n{'='*55}")
    print(f"RUN {run_no} | SEED {seed}")
    print("Selected files:", selected_files)
    print("pricing.py included:", bug_file_found)
    print("Context characters:", len(context))
    print("\nGemma response:\n")
    print(response)

print("\nTotal completed runs:", len(random_experiment_results))


RUN 1 | SEED 7
Selected files: ['orders.py', 'inventory.py', 'pricing.py', 'validation.py']
pricing.py included: True
Context characters: 686

Gemma response:

Here is the analysis of the provided Python files:

### Analysis Summary

After reviewing all the provided files (`orders.py`, `inventory.py`, `pricing.py`, `validation.py`), **no obvious runtime errors or logical bugs are present based solely on the provided code structure and logic.**

Each function appears to perform its intended calculation or retrieval based on the inputs it receives.

*   `orders.py`: Calls functions correctly.
*   `inventory.py`: Returns stock levels (defaults to 0 if product is unknown).
*   `pricing.py`: Calculates total correctly ($\text{Subtotal} - (\text{Subtotal} \times \text{Discount Rate})$).
*   `validation.py`: Checks if quantity is positive.

**Conclusion:** Based strictly on the provided code snippets, I cannot identify an actual bug.

---
### Formal Answer Structure (Addressing the Task Requ

In [50]:
random_summary = {
    "total_runs": 5,
    "bug_file_retrieved": 3,
    "bug_file_missed": 2,
    "retrieval_hit_rate": 0.60,
    "correct_bug_localization": 0,
    "localization_accuracy": 0.0
}

print(random_summary)

{'total_runs': 5, 'bug_file_retrieved': 3, 'bug_file_missed': 2, 'retrieval_hit_rate': 0.6, 'correct_bug_localization': 0, 'localization_accuracy': 0.0}


In [51]:
import json

with open("random_retrieval_results.json", "w") as f:
    json.dump(
        {
            "summary": random_summary,
            "runs": random_experiment_results
        },
        f,
        indent=4
    )

print("Experiment results saved successfully!")

Experiment results saved successfully!


In [53]:
def create_evaluation_prompt(context):
    return f"""
You are a repository-level Python bug localization agent.

Analyze the following Python repository files.

REPOSITORY CODE:
{context}

TASK:
1. Identify the single most likely buggy file.
2. Identify the function containing the bug.
3. Identify the exact faulty statement.
4. Explain the root cause using code evidence.
5. Provide a minimal correction.

RULES:
- A design improvement is not necessarily a bug.
- Do not invent missing code.
- Trace function calls and dependencies when possible.
- If evidence is insufficient, explicitly state uncertainty.
- Do not assume code is correct without checking its logic.
"""

print("Common evaluation prompt created.")

Common evaluation prompt created.


In [54]:
# Graph retrieval context
graph_context = retrieved_context

# Random retrieval context
random_context_control = random_context

print("Graph context:", len(graph_context), "characters")
print("Random context:", len(random_context_control), "characters")

print("\nGraph files:", relevant_files)
print("Random files:", random_files)

Graph context: 704 characters
Random context: 414 characters

Graph files: ['discounts.py', 'main.py', 'orders.py', 'pricing.py']
Random files: ['discounts.py', 'catalog.py', 'notifications.py', 'validation.py']


In [55]:
def run_evaluation(context, method_name):

    prompt = create_evaluation_prompt(context)

    messages = [{"role": "user", "content": prompt}]

    formatted_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.1,
            use_cache=True
        )

    response = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    print(f"\n{'='*50}")
    print("METHOD:", method_name)
    print("Input tokens:", inputs["input_ids"].shape[-1])
    print("\nRESPONSE:\n")
    print(response)

    return response

In [56]:
graph_result = run_evaluation(
    graph_context,
    "Graph Retrieval"
)


METHOD: Graph Retrieval
Input tokens: 348

RESPONSE:

## Bug Localization Analysis

**1. Identify the single most likely buggy file.**
`orders.py` (or potentially `discounts.py`, but tracing suggests an issue in how the returned value is used).

**2. Identify the function containing the bug.**
`place_order` in `orders.py`.

**3. Identify the exact faulty statement.**
```python
    return calculate_total(
        unit_price, quantity, discount_rate
    )
```
*(Self-Correction/Refinement: While this line itself looks syntactically correct based on the inputs, the bug lies in the assumption about what `discount_rate` represents, which is determined by `get_discount`.)*

Let's re-examine the flow to find the logical error:

*   `main.py` calls `place_order("BOOK", 2, "SAVE10")`.
*   `orders.py:place_order` calls `get_discount("SAVE10")`.
*   `dis


In [57]:
random_result = run_evaluation(
    random_context_control,
    "Random Retrieval"
)


METHOD: Random Retrieval
Input tokens: 271

RESPONSE:

## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`discounts.py`

**2. Identify the function containing the bug:**
`get_discount(coupon)`

**3. Identify the exact faulty statement:**
```python
    if coupon == "SAVE10":
        return 10
    return 0
```
*(Self-Correction/Refinement: While this function works as written for the specific inputs provided, in a real-world scenario where discount logic is expected to be more robust or handle other cases, returning `0` for all non-matching coupons might be considered incomplete or potentially buggy if the expectation is that an invalid coupon should raise an error or return a specific default value other than 0.)*

**Re-evaluation based on strict interpretation of "bug":**
The provided code snippet in `discounts.py` is logically sound for the defined behavior (if coupon is "SAVE10", return 10; otherwise, return 0). Without external context defining what c

In [58]:
comparison_results = {
    "graph": {
        "files": relevant_files,
        "context_chars": len(graph_context),
        "correct_localization": False,
        "predicted_file": "orders.py"
    },
    "random": {
        "files": random_files,
        "context_chars": len(random_context_control),
        "correct_localization": False,
        "predicted_file": "discounts.py"
    }
}

print(comparison_results)

{'graph': {'files': ['discounts.py', 'main.py', 'orders.py', 'pricing.py'], 'context_chars': 704, 'correct_localization': False, 'predicted_file': 'orders.py'}, 'random': {'files': ['discounts.py', 'catalog.py', 'notifications.py', 'validation.py'], 'context_chars': 414, 'correct_localization': False, 'predicted_file': 'discounts.py'}}


In [59]:
import json

with open("experiment_5_comparison.json", "w") as f:
    json.dump(comparison_results, f, indent=4)

print("Experiment 5 saved successfully!")

Experiment 5 saved successfully!


In [60]:
from pathlib import Path
import shutil

benchmark_root = Path("benchmark_tasks")
task1 = benchmark_root / "task_1_discount_bug"

task1.mkdir(parents=True, exist_ok=True)

task1_files = [
    "main.py",
    "orders.py",
    "discounts.py",
    "pricing.py"
]

for filename in task1_files:
    shutil.copy2(project / filename, task1 / filename)

print("Task 1 benchmark created!")
print("\nFiles:")

for file in sorted(task1.glob("*.py")):
    print("-", file.name)

Task 1 benchmark created!

Files:
- discounts.py
- main.py
- orders.py
- pricing.py


In [61]:
import sys

sys.path.insert(0, str(task1.resolve()))

from pricing import calculate_total

actual = calculate_total(100, 2, 10)

print("Input:")
print("Unit price: 100")
print("Quantity: 2")
print("Discount rate: 10%")

print("\nExpected result:", 180)
print("Actual result:", actual)

assert actual != 180, "Bug was not reproduced!"
print("\nBug successfully reproduced!")

Input:
Unit price: 100
Quantity: 2
Discount rate: 10%

Expected result: 180
Actual result: -1800

Bug successfully reproduced!


In [62]:
task1_test_cases = [
    (100, 2, 10, 180),
    (200, 3, 20, 480),
    (50, 4, 0, 200),
    (100, 1, 100, 0),
]

print("Task 1 Test Cases:\n")

for price, quantity, rate, expected in task1_test_cases:
    print({
        "unit_price": price,
        "quantity": quantity,
        "discount_rate": rate,
        "expected": expected
    })

print("\nTotal test cases:", len(task1_test_cases))

Task 1 Test Cases:

{'unit_price': 100, 'quantity': 2, 'discount_rate': 10, 'expected': 180}
{'unit_price': 200, 'quantity': 3, 'discount_rate': 20, 'expected': 480}
{'unit_price': 50, 'quantity': 4, 'discount_rate': 0, 'expected': 200}
{'unit_price': 100, 'quantity': 1, 'discount_rate': 100, 'expected': 0}

Total test cases: 4


In [63]:
import importlib.util

# Task 1-এর pricing.py আলাদাভাবে load করছি
# কেন? অন্য কোনো pricing.py আগে import করা থাকলে
# যেন ভুল module ব্যবহার না হয়।
pricing_path = task1 / "pricing.py"

spec = importlib.util.spec_from_file_location(
    "task1_pricing",
    pricing_path
)

task1_pricing = importlib.util.module_from_spec(spec)
spec.loader.exec_module(task1_pricing)

# Run all test cases
passed = 0
failed = 0

for price, quantity, rate, expected in task1_test_cases:

    actual = task1_pricing.calculate_total(
        price, quantity, rate
    )

    if actual == expected:
        passed += 1
        status = "PASS"
    else:
        failed += 1
        status = "FAIL"

    print(
        f"{status} | Input: ({price}, {quantity}, {rate}) "
        f"| Expected: {expected} | Actual: {actual}"
    )

print("\nTest Summary")
print("Passed:", passed)
print("Failed:", failed)
print("Total:", len(task1_test_cases))

FAIL | Input: (100, 2, 10) | Expected: 180 | Actual: -1800
FAIL | Input: (200, 3, 20) | Expected: 480 | Actual: -11400
PASS | Input: (50, 4, 0) | Expected: 200 | Actual: 200
FAIL | Input: (100, 1, 100) | Expected: 0 | Actual: -9900

Test Summary
Passed: 1
Failed: 3
Total: 4


In [64]:
import json

task1_ground_truth = {
    "task_id": "task_1_discount_bug",
    "buggy_file": "pricing.py",
    "buggy_function": "calculate_total",
    "faulty_statement": "discount = subtotal * discount_rate",
    "root_cause": "Discount percentage is not divided by 100.",
    "correct_statement": "discount = subtotal * (discount_rate / 100)",
    "tests": {
        "total": 4,
        "passed": 1,
        "failed": 3
    }
}

with open(task1 / "ground_truth.json", "w") as f:
    json.dump(task1_ground_truth, f, indent=4)

print("Task 1 ground truth saved!")

Task 1 ground truth saved!


In [65]:
import json

task1_ground_truth = {
    "task_id": "task_1_discount_bug",
    "buggy_file": "pricing.py",
    "buggy_function": "calculate_total",
    "faulty_statement": "discount = subtotal * discount_rate",
    "root_cause": "Discount percentage is not divided by 100.",
    "correct_statement": "discount = subtotal * (discount_rate / 100)",
    "tests": {
        "total": 4,
        "passed": 1,
        "failed": 3
    }
}

with open(task1 / "ground_truth.json", "w") as f:
    json.dump(task1_ground_truth, f, indent=4)

print("Task 1 ground truth saved!")

Task 1 ground truth saved!


In [66]:
from pathlib import Path

task2 = benchmark_root / "task_2_inventory_bug"
task2.mkdir(parents=True, exist_ok=True)

print("Task 2 folder created:", task2)

Task 2 folder created: benchmark_tasks/task_2_inventory_bug


In [67]:
from pathlib import Path

task2 = benchmark_root / "task_2_inventory_bug"
task2.mkdir(parents=True, exist_ok=True)

print("Task 2 folder created:", task2)

Task 2 folder created: benchmark_tasks/task_2_inventory_bug


In [68]:
task2_files = {
    "main.py": '''
from orders import place_order

def run():
    return place_order("BOOK", 5)
''',

    "orders.py": '''
from inventory import check_stock

def place_order(product_id, quantity):
    if not check_stock(product_id, quantity):
        return "Out of stock"

    return "Order successful"
''',

    "inventory.py": '''
def check_stock(product_id, quantity):
    stock = {"BOOK": 50}
    available = stock.get(product_id, 0)

    # BUG: Wrong comparison operator
    return available < quantity
'''
}

for filename, content in task2_files.items():
    (task2 / filename).write_text(content.strip() + "\n")

print("Task 2 repository created!")

for file in sorted(task2.glob("*.py")):
    print("-", file.name)

Task 2 repository created!
- inventory.py
- main.py
- orders.py


In [69]:
import importlib.util

inventory_path = task2 / "inventory.py"

spec = importlib.util.spec_from_file_location(
    "task2_inventory",
    inventory_path
)

task2_inventory = importlib.util.module_from_spec(spec)
spec.loader.exec_module(task2_inventory)

actual = task2_inventory.check_stock("BOOK", 5)

expected = True

print("Available stock: 50")
print("Requested quantity: 5")
print("Expected result:", expected)
print("Actual result:", actual)

assert actual != expected, "Bug was not reproduced!"

print("\nTask 2 bug successfully reproduced!")

Available stock: 50
Requested quantity: 5
Expected result: True
Actual result: False

Task 2 bug successfully reproduced!


In [70]:
task2_test_cases = [
    ("BOOK", 5, True),     # Stock available
    ("BOOK", 50, True),    # Exact stock available
    ("BOOK", 60, False),   # Insufficient stock
    ("PEN", 5, False),     # Unknown product
]

print("Task 2 Test Cases:\n")

for product_id, quantity, expected in task2_test_cases:
    print({
        "product_id": product_id,
        "quantity": quantity,
        "expected": expected
    })

print("\nTotal test cases:", len(task2_test_cases))

Task 2 Test Cases:

{'product_id': 'BOOK', 'quantity': 5, 'expected': True}
{'product_id': 'BOOK', 'quantity': 50, 'expected': True}
{'product_id': 'BOOK', 'quantity': 60, 'expected': False}
{'product_id': 'PEN', 'quantity': 5, 'expected': False}

Total test cases: 4


In [71]:
passed = 0
failed = 0

for product_id, quantity, expected in task2_test_cases:

    actual = task2_inventory.check_stock(
        product_id, quantity
    )

    if actual == expected:
        status = "PASS"
        passed += 1
    else:
        status = "FAIL"
        failed += 1

    print(
        f"{status} | Product: {product_id} "
        f"| Quantity: {quantity} "
        f"| Expected: {expected} "
        f"| Actual: {actual}"
    )

print("\nTask 2 Test Summary")
print("Passed:", passed)
print("Failed:", failed)
print("Total:", len(task2_test_cases))

FAIL | Product: BOOK | Quantity: 5 | Expected: True | Actual: False
FAIL | Product: BOOK | Quantity: 50 | Expected: True | Actual: False
FAIL | Product: BOOK | Quantity: 60 | Expected: False | Actual: True
FAIL | Product: PEN | Quantity: 5 | Expected: False | Actual: True

Task 2 Test Summary
Passed: 0
Failed: 4
Total: 4


In [72]:
import json

task2_ground_truth = {
    "task_id": "task_2_inventory_bug",
    "buggy_file": "inventory.py",
    "buggy_function": "check_stock",
    "faulty_statement": "return available < quantity",
    "root_cause": "The comparison operator is reversed. The function returns True when stock is insufficient instead of when stock is sufficient.",
    "correct_statement": "return available >= quantity",
    "tests": {
        "total": 4,
        "passed": 0,
        "failed": 4
    }
}

with open(task2 / "ground_truth.json", "w") as f:
    json.dump(task2_ground_truth, f, indent=4)

print("Task 2 ground truth saved!")

Task 2 ground truth saved!


In [73]:
task3 = benchmark_root / "task_3_boundary_bug"
task3.mkdir(parents=True, exist_ok=True)

print("Task 3 folder created:", task3)

Task 3 folder created: benchmark_tasks/task_3_boundary_bug


In [74]:
task3_files = {
    "main.py": '''
from validation import validate_quantity

def run():
    return validate_quantity(100)
''',

    "orders.py": '''
from validation import validate_quantity

def place_order(quantity):
    if not validate_quantity(quantity):
        return "Invalid quantity"

    return "Order accepted"
''',

    "validation.py": '''
def validate_quantity(quantity):
    # BUG: Boundary condition is incorrect
    return quantity > 0 and quantity < 100
'''
}

for filename, content in task3_files.items():
    (task3 / filename).write_text(content.strip() + "\n")

print("Task 3 repository created!")

for file in sorted(task3.glob("*.py")):
    print("-", file.name)

Task 3 repository created!
- main.py
- orders.py
- validation.py


In [75]:
import importlib.util

validation_path = task3 / "validation.py"

spec = importlib.util.spec_from_file_location(
    "task3_validation",
    validation_path
)

task3_validation = importlib.util.module_from_spec(spec)
spec.loader.exec_module(task3_validation)

actual = task3_validation.validate_quantity(100)
expected = True

print("Maximum allowed quantity: 100")
print("Test quantity: 100")
print("Expected result:", expected)
print("Actual result:", actual)

assert actual != expected, "Bug was not reproduced!"

print("\nTask 3 bug successfully reproduced!")

Maximum allowed quantity: 100
Test quantity: 100
Expected result: True
Actual result: False

Task 3 bug successfully reproduced!


In [76]:
task3_test_cases = [
    (0, False),     # Invalid: zero
    (1, True),      # Valid: minimum
    (99, True),     # Valid: below maximum
    (100, True),    # Valid: exact maximum
    (101, False),   # Invalid: above maximum
]

print("Task 3 Test Cases:\n")

for quantity, expected in task3_test_cases:
    print({
        "quantity": quantity,
        "expected": expected
    })

print("\nTotal test cases:", len(task3_test_cases))

Task 3 Test Cases:

{'quantity': 0, 'expected': False}
{'quantity': 1, 'expected': True}
{'quantity': 99, 'expected': True}
{'quantity': 100, 'expected': True}
{'quantity': 101, 'expected': False}

Total test cases: 5


In [77]:
passed = 0
failed = 0

for quantity, expected in task3_test_cases:

    actual = task3_validation.validate_quantity(quantity)

    if actual == expected:
        status = "PASS"
        passed += 1
    else:
        status = "FAIL"
        failed += 1

    print(
        f"{status} | Quantity: {quantity} "
        f"| Expected: {expected} "
        f"| Actual: {actual}"
    )

print("\nTask 3 Test Summary")
print("Passed:", passed)
print("Failed:", failed)
print("Total:", len(task3_test_cases))

PASS | Quantity: 0 | Expected: False | Actual: False
PASS | Quantity: 1 | Expected: True | Actual: True
PASS | Quantity: 99 | Expected: True | Actual: True
FAIL | Quantity: 100 | Expected: True | Actual: False
PASS | Quantity: 101 | Expected: False | Actual: False

Task 3 Test Summary
Passed: 4
Failed: 1
Total: 5


In [78]:
import json

task3_ground_truth = {
    "task_id": "task_3_boundary_bug",
    "buggy_file": "validation.py",
    "buggy_function": "validate_quantity",
    "faulty_statement": "return quantity > 0 and quantity < 100",
    "root_cause": "The upper boundary is excluded because the code uses < 100 instead of <= 100.",
    "correct_statement": "return quantity > 0 and quantity <= 100",
    "tests": {
        "total": 5,
        "passed": 4,
        "failed": 1
    }
}

with open(task3 / "ground_truth.json", "w") as f:
    json.dump(task3_ground_truth, f, indent=4)

print("Task 3 ground truth saved!")

Task 3 ground truth saved!


In [80]:
import json

benchmark_tasks = {
    "task_1": benchmark_root / "task_1_discount_bug",
    "task_2": benchmark_root / "task_2_inventory_bug",
    "task_3": benchmark_root / "task_3_boundary_bug"
}

benchmark_data = {}

for task_name, task_path in benchmark_tasks.items():

    with open(task_path / "ground_truth.json", "r") as f:
        ground_truth = json.load(f)

    python_files = sorted(task_path.glob("*.py"))

    benchmark_data[task_name] = {
        "path": task_path,
        "files": python_files,
        "ground_truth": ground_truth
    }

    print(f"\n{task_name}")
    print("Python files:", len(python_files))
    print("Bug file:", ground_truth["buggy_file"])
    print("Bug function:", ground_truth["buggy_function"])
    print("Ground truth: Verified")


task_1
Python files: 4
Bug file: pricing.py
Bug function: calculate_total
Ground truth: Verified

task_2
Python files: 3
Bug file: inventory.py
Bug function: check_stock
Ground truth: Verified

task_3
Python files: 3
Bug file: validation.py
Bug function: validate_quantity
Ground truth: Verified


In [81]:
import ast

def build_dependency_graph(task_path):

    python_files = list(task_path.glob("*.py"))

    file_names = {
        file.stem: file.name
        for file in python_files
    }

    graph = {}

    for file in python_files:

        tree = ast.parse(file.read_text())

        dependencies = set()

        for node in ast.walk(tree):

            if isinstance(node, ast.Import):
                for alias in node.names:
                    if alias.name in file_names:
                        dependencies.add(file_names[alias.name])

            elif isinstance(node, ast.ImportFrom):
                if node.module in file_names:
                    dependencies.add(file_names[node.module])

        graph[file.name] = sorted(dependencies)

    return graph


# Build graphs for all benchmark tasks
dependency_graphs = {}

for task_name, task_info in benchmark_data.items():

    graph = build_dependency_graph(task_info["path"])

    dependency_graphs[task_name] = graph

    print(f"\n{task_name} dependency graph:")

    for file, dependencies in graph.items():
        print(f"{file} -> {dependencies}")


task_1 dependency graph:
main.py -> ['orders.py']
pricing.py -> []
orders.py -> ['discounts.py', 'pricing.py']
discounts.py -> []

task_2 dependency graph:
inventory.py -> []
main.py -> ['orders.py']
orders.py -> ['inventory.py']

task_3 dependency graph:
validation.py -> []
main.py -> ['validation.py']
orders.py -> ['validation.py']


In [82]:
from collections import deque

def retrieve_related_files(start_file, graph):

    visited = set()
    queue = deque([start_file])

    while queue:

        current = queue.popleft()

        if current in visited:
            continue

        visited.add(current)

        for dependency in graph.get(current, []):
            if dependency not in visited:
                queue.append(dependency)

    return sorted(visited)


# Test all benchmark tasks
for task_name, graph in dependency_graphs.items():

    retrieved = retrieve_related_files("main.py", graph)

    print(f"\n{task_name}")
    print("Retrieved files:", retrieved)
    print("Retrieved count:", len(retrieved))
    print("Total files:", len(graph))


task_1
Retrieved files: ['discounts.py', 'main.py', 'orders.py', 'pricing.py']
Retrieved count: 4
Total files: 4

task_2
Retrieved files: ['inventory.py', 'main.py', 'orders.py']
Retrieved count: 3
Total files: 3

task_3
Retrieved files: ['main.py', 'validation.py']
Retrieved count: 2
Total files: 3


In [83]:
distractor_modules = {
    "logger.py": """
def log_message(message):
    return f"LOG: {message}"
""",

    "email_service.py": """
def send_email(recipient, message):
    return {
        "recipient": recipient,
        "message": message
    }
""",

    "reporting.py": """
def generate_report(items):
    return len(items)
""",

    "backup.py": """
def create_backup(filename):
    return f"Backup created for {filename}"
""",

    "analytics.py": """
def calculate_average(numbers):
    if not numbers:
        return 0
    return sum(numbers) / len(numbers)
"""
}

for task_name, task_info in benchmark_data.items():

    task_path = task_info["path"]

    for filename, code in distractor_modules.items():
        file_path = task_path / filename
        file_path.write_text(code.strip())

    print(f"{task_name}: 5 distractor files added")
    print("Total Python files:", len(list(task_path.glob("*.py"))))

task_1: 5 distractor files added
Total Python files: 9
task_2: 5 distractor files added
Total Python files: 8
task_3: 5 distractor files added
Total Python files: 8


In [84]:
# Rebuild dependency graphs after adding distractor files

dependency_graphs = {}

for task_name, task_info in benchmark_data.items():

    graph = build_dependency_graph(task_info["path"])
    dependency_graphs[task_name] = graph

    print(f"\n{task_name}")
    print("Total files in graph:", len(graph))

    for file, dependencies in sorted(graph.items()):
        print(f"{file} -> {dependencies}")


task_1
Total files in graph: 9
analytics.py -> []
backup.py -> []
discounts.py -> []
email_service.py -> []
logger.py -> []
main.py -> ['orders.py']
orders.py -> ['discounts.py', 'pricing.py']
pricing.py -> []
reporting.py -> []

task_2
Total files in graph: 8
analytics.py -> []
backup.py -> []
email_service.py -> []
inventory.py -> []
logger.py -> []
main.py -> ['orders.py']
orders.py -> ['inventory.py']
reporting.py -> []

task_3
Total files in graph: 8
analytics.py -> []
backup.py -> []
email_service.py -> []
logger.py -> []
main.py -> ['validation.py']
orders.py -> ['validation.py']
reporting.py -> []
validation.py -> []


In [85]:
retrieval_results = {}

for task_name, graph in dependency_graphs.items():

    retrieved = retrieve_related_files("main.py", graph)

    retrieval_results[task_name] = retrieved

    print(f"\n{task_name}")
    print("Retrieved files:", retrieved)
    print("Retrieved count:", len(retrieved))
    print("Total files:", len(graph))
    print(
        "Context reduction:",
        round((1 - len(retrieved) / len(graph)) * 100, 2),
        "%"
    )


task_1
Retrieved files: ['discounts.py', 'main.py', 'orders.py', 'pricing.py']
Retrieved count: 4
Total files: 9
Context reduction: 55.56 %

task_2
Retrieved files: ['inventory.py', 'main.py', 'orders.py']
Retrieved count: 3
Total files: 8
Context reduction: 62.5 %

task_3
Retrieved files: ['main.py', 'validation.py']
Retrieved count: 2
Total files: 8
Context reduction: 75.0 %


In [86]:
context_metrics = {}

for task_name, task_info in benchmark_data.items():

    all_files = task_info["files"]
    retrieved_names = retrieval_results[task_name]
    ground_truth = task_info["ground_truth"]

    # Full repository context
    full_context = "\n\n".join(
        f"# FILE: {file.name}\n{file.read_text()}"
        for file in sorted(all_files)
    )

    # Graph-retrieved context
    graph_context = "\n\n".join(
        f"# FILE: {file.name}\n{file.read_text()}"
        for file in sorted(all_files)
        if file.name in retrieved_names
    )

    bug_file = ground_truth["buggy_file"]
    bug_retrieved = bug_file in retrieved_names

    full_chars = len(full_context)
    graph_chars = len(graph_context)

    reduction = (
        (1 - graph_chars / full_chars) * 100
        if full_chars else 0
    )

    context_metrics[task_name] = {
        "bug_file_retrieved": bug_retrieved,
        "full_chars": full_chars,
        "graph_chars": graph_chars,
        "character_reduction": reduction
    }

    print(f"\n{task_name}")
    print("Bug file:", bug_file)
    print("Bug file retrieved:", bug_retrieved)
    print("Full context characters:", full_chars)
    print("Graph context characters:", graph_chars)
    print("Character reduction:", round(reduction, 2), "%")


task_1
Bug file: pricing.py
Bug file retrieved: True
Full context characters: 710
Graph context characters: 710
Character reduction: 0.0 %

task_2
Bug file: inventory.py
Bug file retrieved: True
Full context characters: 492
Graph context characters: 492
Character reduction: 0.0 %

task_3
Bug file: validation.py
Bug file retrieved: True
Full context characters: 438
Graph context characters: 246
Character reduction: 43.84 %


In [87]:
for task_name, task_info in benchmark_data.items():

    print(f"\n{task_name}")

    for file in sorted(task_info["files"]):
        content = file.read_text()

        print(
            f"{file.name:20} "
            f"{len(content):5} characters"
        )


task_1
discounts.py            85 characters
main.py                 88 characters
orders.py              290 characters
pricing.py             167 characters

task_2
inventory.py           175 characters
main.py                 77 characters
orders.py              181 characters

task_3
main.py                 87 characters
orders.py              172 characters
validation.py          119 characters


In [88]:
for task_name, task_info in benchmark_data.items():

    # Refresh all Python files from disk
    task_info["files"] = sorted(
        task_info["path"].glob("*.py")
    )

    print(f"\n{task_name}")
    print("Total files:", len(task_info["files"]))

    for file in task_info["files"]:
        print(f"{file.name:20} {len(file.read_text())} characters")


task_1
Total files: 9
analytics.py         107 characters
backup.py            72 characters
discounts.py         85 characters
email_service.py     113 characters
logger.py            54 characters
main.py              88 characters
orders.py            290 characters
pricing.py           167 characters
reporting.py         49 characters

task_2
Total files: 8
analytics.py         107 characters
backup.py            72 characters
email_service.py     113 characters
inventory.py         175 characters
logger.py            54 characters
main.py              77 characters
orders.py            181 characters
reporting.py         49 characters

task_3
Total files: 8
analytics.py         107 characters
backup.py            72 characters
email_service.py     113 characters
logger.py            54 characters
main.py              87 characters
orders.py            172 characters
reporting.py         49 characters
validation.py        119 characters


In [89]:
context_metrics = {}

for task_name, task_info in benchmark_data.items():

    all_files = task_info["files"]
    retrieved_names = retrieval_results[task_name]
    ground_truth = task_info["ground_truth"]

    # Full repository context
    full_context = "\n\n".join(
        f"# FILE: {file.name}\n{file.read_text()}"
        for file in sorted(all_files)
    )

    # Graph-retrieved context
    graph_context = "\n\n".join(
        f"# FILE: {file.name}\n{file.read_text()}"
        for file in sorted(all_files)
        if file.name in retrieved_names
    )

    bug_file = ground_truth["buggy_file"]

    full_chars = len(full_context)
    graph_chars = len(graph_context)

    reduction = (
        (1 - graph_chars / full_chars) * 100
        if full_chars else 0
    )

    context_metrics[task_name] = {
        "bug_file_retrieved": bug_file in retrieved_names,
        "full_chars": full_chars,
        "graph_chars": graph_chars,
        "character_reduction": reduction
    }

    print(f"\n{task_name}")
    print("Bug file retrieved:", bug_file in retrieved_names)
    print("Full context characters:", full_chars)
    print("Graph context characters:", graph_chars)
    print("Character reduction:", round(reduction, 2), "%")


task_1
Bug file retrieved: True
Full context characters: 1218
Graph context characters: 710
Character reduction: 41.71 %

task_2
Bug file retrieved: True
Full context characters: 1000
Graph context characters: 492
Character reduction: 50.8 %

task_3
Bug file retrieved: True
Full context characters: 946
Graph context characters: 246
Character reduction: 74.0 %


In [90]:
import random

RANDOM_SEEDS = [11, 22, 33, 44, 55]

evaluation_data = {}

def make_context(files):
    return "\n\n".join(
        f"# FILE: {file.name}\n{file.read_text()}"
        for file in sorted(files)
    )

for task_name, task_info in benchmark_data.items():

    all_files = task_info["files"]
    graph_names = set(retrieval_results[task_name])

    graph_files = [
        file for file in all_files
        if file.name in graph_names
    ]

    # Strategy A: Full repository
    full_context = make_context(all_files)

    # Strategy B: Graph retrieval
    graph_context = make_context(graph_files)

    # Strategy C: Random retrieval with equal file count
    random_contexts = {}

    for seed in RANDOM_SEEDS:
        rng = random.Random(seed)

        sampled_files = rng.sample(
            all_files,
            len(graph_files)
        )

        random_contexts[seed] = {
            "files": [file.name for file in sampled_files],
            "context": make_context(sampled_files)
        }

    evaluation_data[task_name] = {
        "full": full_context,
        "graph": graph_context,
        "random": random_contexts
    }

    bug_file = task_info["ground_truth"]["buggy_file"]

    print(f"\n{task_name}")
    print("Full files:", len(all_files))
    print("Graph files:", len(graph_files))
    print("Graph bug-file hit:", bug_file in graph_names)

    for seed, result in random_contexts.items():
        hit = bug_file in result["files"]

        print(
            f"Random seed {seed}: "
            f"{len(result['files'])} files | "
            f"Bug file hit: {hit}"
        )


task_1
Full files: 9
Graph files: 4
Graph bug-file hit: True
Random seed 11: 4 files | Bug file hit: True
Random seed 22: 4 files | Bug file hit: False
Random seed 33: 4 files | Bug file hit: False
Random seed 44: 4 files | Bug file hit: True
Random seed 55: 4 files | Bug file hit: False

task_2
Full files: 8
Graph files: 3
Graph bug-file hit: True
Random seed 11: 3 files | Bug file hit: False
Random seed 22: 3 files | Bug file hit: False
Random seed 33: 3 files | Bug file hit: False
Random seed 44: 3 files | Bug file hit: False
Random seed 55: 3 files | Bug file hit: False

task_3
Full files: 8
Graph files: 2
Graph bug-file hit: True
Random seed 11: 2 files | Bug file hit: True
Random seed 22: 2 files | Bug file hit: False
Random seed 33: 2 files | Bug file hit: False
Random seed 44: 2 files | Bug file hit: False
Random seed 55: 2 files | Bug file hit: True


In [91]:
import json
import torch

def create_evaluation_prompt(context):

    return f"""
You are a repository-level Python bug localization agent.

Analyze the following Python repository files.

REPOSITORY CODE:
{context}

TASK:
1. Identify the single most likely buggy file.
2. Identify the function containing the bug.
3. Identify the exact faulty statement.
4. Explain the root cause using code evidence.
5. Provide a minimal correction.

RULES:
- A design improvement is not necessarily a bug.
- Do not invent missing code.
- Trace function calls and dependencies when possible.
- If evidence is insufficient, explicitly state uncertainty.
- Do not assume code is correct without checking its logic.
"""

def evaluate_gemma(context):

    prompt = create_evaluation_prompt(context)

    messages = [
        {"role": "user", "content": prompt}
    ]

    formatted_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    input_tokens = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.1,
            use_cache=True
        )

    response = processor.decode(
        outputs[0][input_tokens:],
        skip_special_tokens=True
    )

    return {
        "response": response,
        "input_tokens": int(input_tokens)
    }


# Run all evaluation conditions
model_results = {}

for task_name, strategies in evaluation_data.items():

    model_results[task_name] = {}

    conditions = {
        "full": strategies["full"],
        "graph": strategies["graph"]
    }

    for seed, result in strategies["random"].items():
        conditions[f"random_{seed}"] = result["context"]

    for method_name, context in conditions.items():

        print(f"\nRunning: {task_name} | {method_name}")

        result = evaluate_gemma(context)

        model_results[task_name][method_name] = result

        print("Input tokens:", result["input_tokens"])
        print("Response:")
        print(result["response"])

# Save raw results
with open("experiment_7_model_results.json", "w") as f:
    json.dump(model_results, f, indent=2)

print("\nAll 21 evaluations completed.")
print("Results saved: experiment_7_model_results.json")


Running: task_1 | full
Input tokens: 502
Response:
## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`orders.py`

**2. Identify the function containing the bug:**
`place_order`

**3. Identify the exact faulty statement:**
```python
    return calculate_total(
        unit_price, quantity, discount_rate
    )
```
*(Self-Correction/Refinement: While this line itself isn't syntactically wrong, the issue lies in how `calculate_total` is being called based on the inputs derived from other functions.)*

**4. Explain the root cause using code evidence:**

The bug stems from an incorrect assumption about the return value of `get_discount(coupon)` when calculating the final total.

*   **In `discounts.py`:**
    ```python
    def get_discount(coupon):
        if coupon == "SAVE10":
            return 10  # Returns the integer 10
        return 0
    ```
*   **In `

Running: task_1 | graph
Input tokens: 352
Response:
## Bug Localization Analysis

**1. Identify the

In [92]:
import json

# Preserve the original 21 evaluation results
with open("experiment_7_initial_results.json", "w") as f:
    json.dump(model_results, f, indent=2)

print("Original results preserved.")

# Find comments that may reveal the bug
for task_name, task_info in benchmark_data.items():

    print(f"\n{task_name}")

    for file in task_info["files"]:

        for line_number, line in enumerate(
            file.read_text().splitlines(), start=1
        ):

            if "BUG" in line.upper():
                print(
                    f"{file.name}:{line_number}: {line.strip()}"
                )

Original results preserved.

task_1

task_2
inventory.py:5: # BUG: Wrong comparison operator

task_3
validation.py:2: # BUG: Boundary condition is incorrect


In [93]:
import re

for task_name, task_info in benchmark_data.items():

    for file in task_info["files"]:

        original_code = file.read_text()

        cleaned_lines = []

        for line in original_code.splitlines():

            if re.search(r"#\s*BUG\b", line, re.IGNORECASE):
                continue

            cleaned_lines.append(line)

        cleaned_code = "\n".join(cleaned_lines)

        if original_code.endswith("\n"):
            cleaned_code += "\n"

        file.write_text(cleaned_code)

    print(f"{task_name}: benchmark cleaned")

task_1: benchmark cleaned
task_2: benchmark cleaned
task_3: benchmark cleaned


In [94]:
for task_name, task_info in benchmark_data.items():

    print(f"\n{task_name}")

    bug_file = task_info["ground_truth"]["buggy_file"]
    target_path = task_info["path"] / bug_file

    print("Bug file:", bug_file)
    print("Updated code:")
    print(target_path.read_text())

    assert not any(
        re.search(r"#\s*BUG\b", line, re.IGNORECASE)
        for file in task_info["files"]
        for line in file.read_text().splitlines()
    )

print("\nNo explicit BUG comments found.")


task_1
Bug file: pricing.py
Updated code:

def calculate_total(unit_price, quantity, discount_rate):
    subtotal = unit_price * quantity
    discount = subtotal * discount_rate
    return subtotal - discount


task_2
Bug file: inventory.py
Updated code:
def check_stock(product_id, quantity):
    stock = {"BOOK": 50}
    available = stock.get(product_id, 0)

    return available < quantity


task_3
Bug file: validation.py
Updated code:
def validate_quantity(quantity):
    return quantity > 0 and quantity < 100


No explicit BUG comments found.


In [95]:
clean_evaluation_data = {}

for task_name, task_info in benchmark_data.items():

    all_files = task_info["files"]

    graph_names = set(retrieval_results[task_name])

    graph_files = [
        file for file in all_files
        if file.name in graph_names
    ]

    full_context = make_context(all_files)
    graph_context = make_context(graph_files)

    random_contexts = {}

    for seed in RANDOM_SEEDS:
        random_contexts[seed] = make_context(
            [
                file for file in all_files
                if file.name in evaluation_data[task_name]
                    ["random"][seed]["files"]
            ]
        )

    clean_evaluation_data[task_name] = {
        "full": full_context,
        "graph": graph_context,
        "random": random_contexts
    }

    print(f"\n{task_name}")
    print("Full context:", len(full_context), "characters")
    print("Graph context:", len(graph_context), "characters")


task_1
Full context: 1218 characters
Graph context: 710 characters

task_2
Full context: 963 characters
Graph context: 455 characters

task_3
Full context: 903 characters
Graph context: 203 characters


In [96]:
clean_model_results = {}

for task_name, strategies in clean_evaluation_data.items():

    clean_model_results[task_name] = {}

    conditions = {
        "full": strategies["full"],
        "graph": strategies["graph"]
    }

    for seed, context in strategies["random"].items():
        conditions[f"random_{seed}"] = context

    for method_name, context in conditions.items():

        print(f"\nRunning: {task_name} | {method_name}")

        result = evaluate_gemma(context)

        clean_model_results[task_name][method_name] = result

        print("Input tokens:", result["input_tokens"])
        print("Response:")
        print(result["response"])


# Save clean evaluation results
with open("experiment_7_clean_results.json", "w") as f:
    json.dump(clean_model_results, f, indent=2)

print("\nAll clean evaluations completed.")
print("Results saved: experiment_7_clean_results.json")


Running: task_1 | full
Input tokens: 502
Response:
## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`orders.py`

**2. Identify the function containing the bug:**
`place_order`

**3. Identify the exact faulty statement:**
```python
    return calculate_total(
        unit_price, quantity, discount_rate
    )
```
*(Self-Correction/Refinement: While this line itself isn't syntactically wrong, the issue lies in how `calculate_total` is being called based on the inputs derived from other functions.)*

**4. Explain the root cause using code evidence:**

The bug stems from an incorrect assumption about the return value of `get_discount(coupon)` when calculating the final total.

*   **In `discounts.py`:**
    ```python
    def get_discount(coupon):
        if coupon == "SAVE10":
            return 10  # Returns the integer 10
        return 0
    ```
*   **In `

Running: task_1 | graph
Input tokens: 352
Response:
## Bug Localization Analysis

**1. Identify the

In [97]:
import pandas as pd

scoring_rows = []

for task_name, methods in clean_model_results.items():

    ground_truth = benchmark_data[task_name]["ground_truth"]

    for method_name, result in methods.items():

        scoring_rows.append({
            "task": task_name,
            "method": method_name,
            "input_tokens": result["input_tokens"],
            "actual_buggy_file": ground_truth["buggy_file"],
            "actual_function": ground_truth["buggy_function"],
            "actual_faulty_statement": ground_truth["faulty_statement"],
            "predicted_buggy_file": "",
            "predicted_function": "",
            "predicted_faulty_statement": "",
            "file_correct": "",
            "function_correct": "",
            "statement_correct": "",
            "root_cause_correct": "",
            "response": result["response"]
        })

scoring_df = pd.DataFrame(scoring_rows)

print("Total evaluation records:", len(scoring_df))
display(scoring_df)

Total evaluation records: 21


,task,method,input_tokens,actual_buggy_file,actual_function,actual_faulty_statement,predicted_buggy_file,predicted_function,predicted_faulty_statement,file_correct,function_correct,statement_correct,root_cause_correct,response
0,task_1,full,502,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
1,task_1,graph,352,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
2,task_1,random_11,276,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,"Based on the provided repository code, I will ..."
3,task_1,random_22,278,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
4,task_1,random_33,328,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
5,task_1,random_44,346,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
6,task_1,random_55,262,pricing.py,calculate_total,discount = subtotal * discount_rate,,,,,,,,"Based on the provided repository code, here is..."
7,task_2,full,426,inventory.py,check_stock,return available < quantity,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
8,task_2,graph,276,inventory.py,check_stock,return available < quantity,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...
9,task_2,random_11,242,inventory.py,check_stock,return available < quantity,,,,,,,,## Bug Localization Analysis\n\n**1. Identify ...


In [98]:
# Verify ground truth for all benchmark tasks
for task_name, task_info in benchmark_data.items():
    gt = task_info["ground_truth"]

    print(f"\n{task_name}")
    print("Buggy file:", gt["buggy_file"])
    print("Function:", gt["buggy_function"])
    print("Faulty statement:", gt["faulty_statement"])

# Verify evaluation record counts
print("\nRecords per task:")
print(scoring_df.groupby("task")["method"].count())

print("\nTotal records:", len(scoring_df))

# Inspect the final five records
display(
    scoring_df[
        ["task", "method", "actual_buggy_file", "actual_function"]
    ].tail(5)
)


task_1
Buggy file: pricing.py
Function: calculate_total
Faulty statement: discount = subtotal * discount_rate

task_2
Buggy file: inventory.py
Function: check_stock
Faulty statement: return available < quantity

task_3
Buggy file: validation.py
Function: validate_quantity
Faulty statement: return quantity > 0 and quantity < 100

Records per task:
task
task_1    7
task_2    7
task_3    7
Name: method, dtype: int64

Total records: 21


,task,method,actual_buggy_file,actual_function
16,task_3,random_11,validation.py,validate_quantity
17,task_3,random_22,validation.py,validate_quantity
18,task_3,random_33,validation.py,validate_quantity
19,task_3,random_44,validation.py,validate_quantity
20,task_3,random_55,validation.py,validate_quantity


In [99]:
print("Total records:", len(scoring_df))

print("\nRecords per task:")
print(scoring_df.groupby("task").size().to_string())

print("\nMethods per task:")
for task, group in scoring_df.groupby("task"):
    print(task, sorted(group["method"].tolist()))

print("\nLast 5 records:")
print(
    scoring_df[
        ["task", "method", "actual_buggy_file", "actual_function"]
    ].tail(5).to_string(index=False)
)

Total records: 21

Records per task:
task
task_1    7
task_2    7
task_3    7

Methods per task:
task_1 ['full', 'graph', 'random_11', 'random_22', 'random_33', 'random_44', 'random_55']
task_2 ['full', 'graph', 'random_11', 'random_22', 'random_33', 'random_44', 'random_55']
task_3 ['full', 'graph', 'random_11', 'random_22', 'random_33', 'random_44', 'random_55']

Last 5 records:
  task    method actual_buggy_file   actual_function
task_3 random_11     validation.py validate_quantity
task_3 random_22     validation.py validate_quantity
task_3 random_33     validation.py validate_quantity
task_3 random_44     validation.py validate_quantity
task_3 random_55     validation.py validate_quantity


In [100]:
for task in ["task_1", "task_2", "task_3"]:
    print("\n" + "=" * 60)
    print(task)

    for method in ["full", "graph"]:
        row = scoring_df[
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        ].iloc[0]

        print(f"\n--- {method.upper()} ---")
        print(row["response"][:1800])


task_1

--- FULL ---
## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`orders.py`

**2. Identify the function containing the bug:**
`place_order`

**3. Identify the exact faulty statement:**
```python
    return calculate_total(
        unit_price, quantity, discount_rate
    )
```
*(Self-Correction/Refinement: While this line itself isn't syntactically wrong, the issue lies in how `calculate_total` is being called based on the inputs derived from other functions.)*

**4. Explain the root cause using code evidence:**

The bug stems from an incorrect assumption about the return value of `get_discount(coupon)` when calculating the final total.

*   **In `discounts.py`:**
    ```python
    def get_discount(coupon):
        if coupon == "SAVE10":
            return 10  # Returns the integer 10
        return 0
    ```
*   **In `

--- GRAPH ---
## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`orders.py`

**2. Identify the fun

In [101]:
for task in ["task_1", "task_2", "task_3"]:
    print("\n" + "=" * 70)
    print(task)

    for method in [
        "random_11",
        "random_22",
        "random_33",
        "random_44",
        "random_55"
    ]:
        row = scoring_df[
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        ].iloc[0]

        print(f"\n--- {method.upper()} ---")
        print(row["response"][:1200])


task_1

--- RANDOM_11 ---
Based on the provided repository code, I will analyze each file for potential bugs.

### Analysis

1.  **`email_service.py`**:
    ```python
    def send_email(recipient, message):
        return {
            "recipient": recipient,
            "message": message
        }
    ```
    This function simply packages recipient and message into a dictionary. It appears logically sound for its stated purpose (returning an email structure).

2.  **`logger.py`**:
    ```python
    def log_message(message):
        return f"LOG: {message}"
    ```
    This function formats a string prefixed with "LOG:". This seems correct for a logging utility.

3.  **`pricing.py`**:
    ```python
    def calculate_total(unit_price, quantity, discount_rate):
        subtotal = unit_price * quantity
        discount = subtotal * discount_rate
        return subtotal - discount
    ```
    This

--- RANDOM_22 ---
## Bug Localization Analysis

**1. Identify the single most likely buggy

In [102]:
predictions = {
    "task_1": {
        "full": "orders.py",
        "graph": "orders.py",
        "random_11": None,
        "random_22": "analytics.py",
        "random_33": "orders.py",
        "random_44": "orders.py",
        "random_55": "main.py"
    },
    "task_2": {
        "full": "inventory.py",
        "graph": "inventory.py",
        "random_11": "orders.py",
        "random_22": "analytics.py",
        "random_33": "main.py",
        "random_44": "orders.py",
        "random_55": None
    },
    "task_3": {
        "full": "validation.py",
        "graph": "validation.py",
        "random_11": "reporting.py",
        "random_22": "email_service.py",
        "random_33": "orders.py",
        "random_44": "main.py",
        "random_55": "validation.py"
    }
}

for task, methods in predictions.items():
    for method, predicted_file in methods.items():

        mask = (
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        )

        idx = scoring_df.index[mask][0]

        scoring_df.loc[idx, "predicted_buggy_file"] = (
            predicted_file if predicted_file else "Uncertain"
        )

        actual = scoring_df.loc[idx, "actual_buggy_file"]

        scoring_df.loc[idx, "file_correct"] = (
            int(predicted_file == actual)
            if predicted_file else None
        )

print("Top-1 scoring completed.")

display(
    scoring_df[
        ["task", "method", "actual_buggy_file",
         "predicted_buggy_file", "file_correct"]
    ]
)

Top-1 scoring completed.


,task,method,actual_buggy_file,predicted_buggy_file,file_correct
0,task_1,full,pricing.py,orders.py,0
1,task_1,graph,pricing.py,orders.py,0
2,task_1,random_11,pricing.py,Uncertain,None
3,task_1,random_22,pricing.py,analytics.py,0
4,task_1,random_33,pricing.py,orders.py,0
5,task_1,random_44,pricing.py,orders.py,0
6,task_1,random_55,pricing.py,main.py,0
7,task_2,full,inventory.py,inventory.py,1
8,task_2,graph,inventory.py,inventory.py,1
9,task_2,random_11,inventory.py,orders.py,0


In [103]:
# Check whether all 21 rows have correct ground truth
for task, info in benchmark_data.items():
    expected_file = info["ground_truth"]["buggy_file"]

    mask = scoring_df["task"] == task
    actual_files = scoring_df.loc[mask, "actual_buggy_file"]

    print(f"\n{task}")
    print("Expected:", expected_file)
    print("Unique actual values:", actual_files.unique())

# Check predictions
print("\nAll prediction records:")
print(
    scoring_df[
        ["task", "method", "actual_buggy_file",
         "predicted_buggy_file", "file_correct"]
    ].to_string(index=False)
)


task_1
Expected: pricing.py
Unique actual values: ['pricing.py']

task_2
Expected: inventory.py
Unique actual values: ['inventory.py']

task_3
Expected: validation.py
Unique actual values: ['validation.py']

All prediction records:
  task    method actual_buggy_file predicted_buggy_file file_correct
task_1      full        pricing.py            orders.py            0
task_1     graph        pricing.py            orders.py            0
task_1 random_11        pricing.py            Uncertain         None
task_1 random_22        pricing.py         analytics.py            0
task_1 random_33        pricing.py            orders.py            0
task_1 random_44        pricing.py            orders.py            0
task_1 random_55        pricing.py              main.py            0
task_2      full      inventory.py         inventory.py            1
task_2     graph      inventory.py         inventory.py            1
task_2 random_11      inventory.py            orders.py            0
task_2 r

In [104]:
summary_rows = []

for method in ["full", "graph"]:
    subset = scoring_df[scoring_df["method"] == method]
    correct = subset["file_correct"].sum()
    total = len(subset)

    summary_rows.append({
        "method": method,
        "correct": int(correct),
        "total": total,
        "accuracy_percent": round(correct / total * 100, 2)
    })

random_subset = scoring_df[
    scoring_df["method"].str.startswith("random")
]

random_scored = random_subset.dropna(subset=["file_correct"])
random_correct = random_scored["file_correct"].sum()

summary_rows.append({
    "method": "random",
    "correct": int(random_correct),
    "total": len(random_scored),
    "accuracy_percent": round(
        random_correct / len(random_scored) * 100, 2
    )
})

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

print(
    "Random strict accuracy (uncertain = incorrect):",
    round(random_correct / len(random_subset) * 100, 2), "%"
)

print(
    "Random prediction coverage:",
    round(len(random_scored) / len(random_subset) * 100, 2), "%"
)

,method,correct,total,accuracy_percent
0,full,2,3,66.67
1,graph,2,3,66.67
2,random,1,13,7.69


Random strict accuracy (uncertain = incorrect): 6.67 %
Random prediction coverage: 86.67 %


In [105]:
for task in ["task_1", "task_2", "task_3"]:
    for method in [
        "full", "graph",
        "random_11", "random_22", "random_33",
        "random_44", "random_55"
    ]:
        row = scoring_df[
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        ].iloc[0]

        print("\n" + "=" * 70)
        print(task, "|", method)
        print(row["response"])


task_1 | full
## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`orders.py`

**2. Identify the function containing the bug:**
`place_order`

**3. Identify the exact faulty statement:**
```python
    return calculate_total(
        unit_price, quantity, discount_rate
    )
```
*(Self-Correction/Refinement: While this line itself isn't syntactically wrong, the issue lies in how `calculate_total` is being called based on the inputs derived from other functions.)*

**4. Explain the root cause using code evidence:**

The bug stems from an incorrect assumption about the return value of `get_discount(coupon)` when calculating the final total.

*   **In `discounts.py`:**
    ```python
    def get_discount(coupon):
        if coupon == "SAVE10":
            return 10  # Returns the integer 10
        return 0
    ```
*   **In `

task_1 | graph
## Bug Localization Analysis

**1. Identify the single most likely buggy file:**
`orders.py`

**2. Identify the function 

In [106]:
# Manually verified function predictions
function_predictions = {
    "task_1": {
        "full": "place_order",
        "graph": "place_order",
        "random_11": None,
        "random_22": "calculate_average",
        "random_33": "place_order",
        "random_44": "place_order",
        "random_55": "run"
    },
    "task_2": {
        "full": "check_stock",
        "graph": "check_stock",
        "random_11": "place_order",
        "random_22": "calculate_average",
        "random_33": "run",
        "random_44": "place_order",
        "random_55": None
    },
    "task_3": {
        "full": "validate_quantity",
        "graph": "validate_quantity",
        "random_11": "generate_report",
        "random_22": "send_email",
        "random_33": "place_order",
        "random_44": "run",
        "random_55": "validate_quantity"
    }
}

# Manually verified exact faulty statement predictions
statement_predictions = {
    "task_1": {
        "full": "return calculate_total(unit_price, quantity, discount_rate)",
        "graph": "discount_rate = get_discount(coupon)",
        "random_11": None,
        "random_22": None,
        "random_33": None,
        "random_44": None,
        "random_55": None
    },
    "task_2": {
        "full": "return available < quantity",
        "graph": "return available < quantity",
        "random_11": "if not check_stock(product_id, quantity):",
        "random_22": None,
        "random_33": "return place_order(\"BOOK\", 5)",
        "random_44": "if not check_stock(product_id, quantity):",
        "random_55": None
    },
    "task_3": {
        "full": "return quantity > 0 and quantity < 100",
        "graph": "return quantity > 0 and quantity < 100",
        "random_11": "return len(items)",
        "random_22": None,
        "random_33": 'return "Order accepted"',
        "random_44": "return validate_quantity(100)",
        "random_55": "return quantity > 0 and quantity < 100"
    }
}

for task, methods in function_predictions.items():
    for method, prediction in methods.items():
        mask = (
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        )
        idx = scoring_df.index[mask][0]
        actual = scoring_df.loc[idx, "actual_function"]

        scoring_df.loc[idx, "predicted_function"] = (
            prediction if prediction else "Uncertain"
        )
        scoring_df.loc[idx, "function_correct"] = (
            int(prediction == actual) if prediction else None
        )

for task, methods in statement_predictions.items():
    for method, prediction in methods.items():
        mask = (
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        )
        idx = scoring_df.index[mask][0]
        actual = scoring_df.loc[idx, "actual_faulty_statement"]

        scoring_df.loc[idx, "predicted_faulty_statement"] = (
            prediction if prediction else "Uncertain"
        )
        scoring_df.loc[idx, "statement_correct"] = (
            int(prediction == actual) if prediction else None
        )

print("Function and statement scoring completed.")

display(
    scoring_df[
        ["task", "method", "function_correct", "statement_correct"]
    ]
)

Function and statement scoring completed.


,task,method,function_correct,statement_correct
0,task_1,full,0,0
1,task_1,graph,0,0
2,task_1,random_11,None,None
3,task_1,random_22,0,None
4,task_1,random_33,0,None
5,task_1,random_44,0,None
6,task_1,random_55,0,None
7,task_2,full,1,1
8,task_2,graph,1,1
9,task_2,random_11,0,0


In [107]:
# Check incomplete and inconsistent scoring records
check_columns = [
    "task",
    "method",
    "predicted_function",
    "function_correct",
    "predicted_faulty_statement",
    "statement_correct"
]

print(scoring_df[check_columns].to_string(index=False))

print("\nMissing values:")
print(scoring_df[check_columns].isna().sum())

print("\nFunction correctness values:")
print(scoring_df["function_correct"].value_counts(dropna=False))

print("\nStatement correctness values:")
print(scoring_df["statement_correct"].value_counts(dropna=False))

  task    method predicted_function function_correct                                  predicted_faulty_statement statement_correct
task_1      full        place_order                0 return calculate_total(unit_price, quantity, discount_rate)                 0
task_1     graph        place_order                0                        discount_rate = get_discount(coupon)                 0
task_1 random_11          Uncertain             None                                                   Uncertain              None
task_1 random_22  calculate_average                0                                                   Uncertain              None
task_1 random_33        place_order                0                                                   Uncertain              None
task_1 random_44        place_order                0                                                   Uncertain              None
task_1 random_55                run                0                               

In [108]:
root_cause_predictions = {
    "task_1": {
        "full": 0,
        "graph": 1,
        "random_11": None,
        "random_22": 0,
        "random_33": 0,
        "random_44": 0,
        "random_55": 0
    },
    "task_2": {
        "full": 1,
        "graph": 1,
        "random_11": 0,
        "random_22": 0,
        "random_33": 0,
        "random_44": 0,
        "random_55": None
    },
    "task_3": {
        "full": 1,
        "graph": 1,
        "random_11": 0,
        "random_22": 0,
        "random_33": 0,
        "random_44": 0,
        "random_55": 0
    }
}

for task, methods in root_cause_predictions.items():
    for method, prediction in methods.items():
        mask = (
            (scoring_df["task"] == task) &
            (scoring_df["method"] == method)
        )
        idx = scoring_df.index[mask][0]

        scoring_df.loc[idx, "root_cause_correct"] = (
            prediction if prediction is not None else None
        )

print("Root cause scoring completed.")

display(
    scoring_df[
        ["task", "method", "root_cause_correct"]
    ]
)

Root cause scoring completed.


,task,method,root_cause_correct
0,task_1,full,0
1,task_1,graph,1
2,task_1,random_11,None
3,task_1,random_22,0
4,task_1,random_33,0
5,task_1,random_44,0
6,task_1,random_55,0
7,task_2,full,1
8,task_2,graph,1
9,task_2,random_11,0


In [109]:
print(
    scoring_df[
        [
            "task",
            "method",
            "root_cause_correct"
        ]
    ].to_string(index=False)
)

print("\nRoot cause score distribution:")
print(
    scoring_df["root_cause_correct"]
    .value_counts(dropna=False)
)

print("\nMissing root cause scores:")
print(
    scoring_df[
        scoring_df["root_cause_correct"].isna()
    ][["task", "method"]]
)

  task    method root_cause_correct
task_1      full                  0
task_1     graph                  1
task_1 random_11               None
task_1 random_22                  0
task_1 random_33                  0
task_1 random_44                  0
task_1 random_55                  0
task_2      full                  1
task_2     graph                  1
task_2 random_11                  0
task_2 random_22                  0
task_2 random_33                  0
task_2 random_44                  0
task_2 random_55               None
task_3      full                  1
task_3     graph                  1
task_3 random_11                  0
task_3 random_22                  0
task_3 random_33                  0
task_3 random_44                  0
task_3 random_55                  0

Root cause score distribution:
root_cause_correct
0       14
1        5
None     2
Name: count, dtype: int64

Missing root cause scores:
      task     method
2   task_1  random_11
13  task_2  random_55


In [110]:
# Create final experiment summary

methods = ["full", "graph", "random"]

metrics = [
    "file_correct",
    "function_correct",
    "statement_correct",
    "root_cause_correct"
]

summary_rows = []

for method in methods:
    subset = scoring_df[
        scoring_df["method"].str.startswith("random")
    ] if method == "random" else scoring_df[
        scoring_df["method"] == method
    ]

    row = {"method": method}

    for metric in metrics:
        evaluated = subset[metric].dropna()
        correct = evaluated.astype(int).sum()

        row[f"{metric}_correct"] = int(correct)
        row[f"{metric}_evaluated"] = len(evaluated)
        row[f"{metric}_accuracy"] = round(
            correct / len(evaluated) * 100, 2
        ) if len(evaluated) else None

    summary_rows.append(row)

final_summary = pd.DataFrame(summary_rows)

display(final_summary)

# Save detailed scores and summary
scoring_df.to_csv(
    "experiment_7_detailed_scoring.csv",
    index=False
)

final_summary.to_csv(
    "experiment_7_final_summary.csv",
    index=False
)

print("Saved:")
print("1. experiment_7_detailed_scoring.csv")
print("2. experiment_7_final_summary.csv")

,method,file_correct_correct,file_correct_evaluated,file_correct_accuracy,function_correct_correct,function_correct_evaluated,function_correct_accuracy,statement_correct_correct,statement_correct_evaluated,statement_correct_accuracy,root_cause_correct_correct,root_cause_correct_evaluated,root_cause_correct_accuracy
0,full,2,3,66.67,2,3,66.67,2,3,66.67,2,3,66.67
1,graph,2,3,66.67,2,3,66.67,2,3,66.67,3,3,100.00
2,random,1,13,7.69,1,13,7.69,1,7,14.29,0,13,0.00


Saved:
1. experiment_7_detailed_scoring.csv
2. experiment_7_final_summary.csv
